# Fitting a stellar spectrum with STARSHIPS

This notebook explains step by step how the stellar spectrum fit works in STARSHIPS.
The goal is to determine the physical parameters of the host star — effective temperature
(Teff), surface gravity (log g), metallicity, projected rotational velocity (vsini),
and systemic radial velocity — by comparing a PHOENIX synthetic stellar spectrum to
the master out-of-transit spectrum measured with SPIRou.

**Why do we fit the stellar spectrum?**

When we analyse the atmosphere of an exoplanet at high spectral resolution, we subtract
the stellar spectrum from the data. If we have a good model of the stellar spectrum,
we can remove stellar lines more accurately, which reduces systematics in the planet
atmosphere analysis. The stellar parameters (especially vsini and Teff) are also
physically interesting in their own right.

**Key challenge: per-order continuum correction**

Each SPIRou spectral order has a different blaze function (the wavelength-dependent
sensitivity of the spectrograph). Even after normalisation, there are residual
continuum shape errors order by order. We correct for this with a polynomial
per order. The clever trick we use is to solve for the best polynomial *analytically*
at each likelihood evaluation, so we never need to include the polynomial parameters
in our sampler. This is called the **profile likelihood** approach.

---

## Contents
1. Setup and data loading
2. Visualising the data
3. The PHOENIX stellar model
4. Rotational broadening
5. The profile likelihood: analytical polynomial per order
6. Running the fit
7. Inspecting the result

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The stellar fit module — all functions and globals live here
import starships.stellar_fit as sf

%matplotlib inline

---
## 1. Setup and data loading

All configuration is read from a YAML file. You should start from
`stellar_fit_inputs_example.yaml` and adapt it to your system.

`setup_stellar_fit()` does everything needed to get ready:
- Reads the YAML config.
- Loads the observed master out-of-transit spectrum.
- Initialises the PHOENIX interpolation grid.
- Pre-computes the wavelength grids for rotational broadening (one per order).
- Pre-computes the Chebyshev polynomial basis matrices (one per order, cached
  for speed since they never change during the fit).

In [ ]:
# Point this to your own YAML config file
CONFIG_FILE = 'my_stellar_fit_config.yaml'

sf.setup_stellar_fit(CONFIG_FILE)

In [ ]:
# After setup, the data is accessible as module-level globals
print(f"Number of orders: {sf.ref_wave.shape[0]}")
print(f"Pixels per order: {sf.ref_wave.shape[1]}")
print(f"Wavelength range: {sf.ref_wave.min():.3f} – {sf.ref_wave.max():.3f} µm")
print(f"Free parameters: {list(sf.params_prior.keys())}")

---
## 2. Visualising the data

Let's first look at a few spectral orders to understand what we are fitting.

In [ ]:
# Plot a selection of orders to see the data quality
orders_to_show = [0, 5, 10, 15, 20]
fig, axes = plt.subplots(len(orders_to_show), 1, figsize=(12, 2.5 * len(orders_to_show)))

for ax, i_ord in zip(axes, orders_to_show):
    wv = sf.ref_wave[i_ord]
    flux = sf.ref_spectrum[i_ord]
    uncert = sf.ref_uncert[i_ord]
    
    ax.plot(wv, flux, 'k', lw=0.6, label='Data')
    ax.fill_between(wv, flux - uncert, flux + uncert, alpha=0.3, color='gray')
    ax.set_ylabel(f'Order #{i_ord}\n{wv[~sf.ref_mask[i_ord]].mean():.3f} µm')
    ax.set_ylim(0, 1.3)

axes[-1].set_xlabel('Wavelength (µm)')
plt.tight_layout()
plt.show()

---
## 3. The PHOENIX stellar model

PHOENIX is a library of synthetic stellar spectra computed for a grid of
(Teff, log g, metallicity) values. We use the `PhoenixInterpGrid` class
to interpolate between grid points.

Let's generate a test spectrum and compare it to the data.

In [ ]:
# Example: build a theta_dict manually with some test values
# (This is what the sampler will do automatically at each step)
theta_test = {
    'teff':    7400.0,   # effective temperature in K
    'logg':    4.3,      # log surface gravity
    'metal':   0.0,      # solar metallicity
    'alpha':   0.0,      # alpha element abundance (fixed)
    'vsini':   86.0,     # projected rotation velocity in km/s
    'epsilon': 0.6,      # linear limb-darkening coefficient
    'v_shift': 0.0,      # systemic RV in km/s
}

# Generate the model for order 10
i_ord_example = 10
model_norm = sf._generate_stellar_model_ord(i_ord_example, **theta_test)

# The model is normalised by its median — same normalisation as the data
print(f"Model median: {np.nanmedian(model_norm):.3f}  (should be close to 1)")

In [ ]:
# Compare the raw model to the data (before polynomial correction)
fig, ax = plt.subplots(figsize=(12, 3))

wv = sf.ref_wave[i_ord_example]
ax.plot(wv, sf.ref_spectrum[i_ord_example], 'k', lw=0.7, label='Data')
ax.plot(wv, model_norm, 'r', lw=0.7, alpha=0.8, label='PHOENIX model (raw, no poly)')

ax.set_xlabel('Wavelength (µm)')
ax.set_ylabel('Normalised flux')
ax.legend()
ax.set_title(f'Order #{i_ord_example} — before polynomial correction')
plt.tight_layout()
plt.show()

print("Notice that the model and data shapes don't match perfectly —")
print("the polynomial correction will fix this.")

---
## 4. Rotational broadening

A star that rotates fast (large vsini) has spectral lines that are broadened
because different parts of the stellar disk have different radial velocities
relative to the observer. We model this with `pyasl.fastRotBroad`.

The broadening is characterised by two parameters:
- `vsini`: the projected equatorial rotation velocity in km/s.
- `epsilon`: the linear limb-darkening coefficient (0 = uniform brightness
  across the disk, 1 = strong limb darkening).

Let's see how changing vsini affects the model.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))

wv = sf.ref_wave[i_ord_example]
ax.plot(wv, sf.ref_spectrum[i_ord_example], 'k', lw=0.7, label='Data', zorder=10)

for vsini_test, color in [(20, 'blue'), (86, 'red'), (130, 'orange')]:
    theta_vsini = dict(theta_test)
    theta_vsini['vsini'] = vsini_test
    model_vsini = sf._generate_stellar_model_ord(i_ord_example, **theta_vsini)
    ax.plot(wv, model_vsini, color=color, lw=0.7, alpha=0.8, label=f'vsini = {vsini_test} km/s')

ax.set_xlabel('Wavelength (µm)')
ax.set_ylabel('Normalised flux')
ax.legend()
ax.set_title('Effect of vsini on the stellar model')
plt.tight_layout()
plt.show()

---
## 5. The profile likelihood: analytical polynomial per order

### Why a polynomial per order?

Even after normalising both the data and the model, their continuum shapes
differ order by order because of imperfect blaze correction. We correct for
this with a Chebyshev polynomial per order.

### Why in log-flux space?

Working in log flux makes the polynomial correction *multiplicative* in flux:

    log(data) ≈ log(model) + polynomial
    ↕
    data ≈ model × exp(polynomial)

This is physically natural — the blaze function multiplies the stellar spectrum.
And crucially, it makes the problem **linear in the polynomial coefficients**,
which means we can solve for them analytically!

### The mathematical solution

For a fixed model and a fixed set of global parameters (Teff, vsini, etc.),
we want to find the polynomial coefficients **c** that minimise:

    χ² = Σ_λ W(λ) [r(λ) − Φ(λ)·c]²

where:
- `r(λ) = log(data) − log(model)` is the residual
- `Φ(λ)` is the row of Chebyshev basis functions at pixel λ
- `W(λ) = 1/σ²_log(λ)` are the weights (`σ_log ≈ σ_flux / flux`)

This is a classic **weighted linear least-squares** problem. The solution is:

    c* = (ΦᵀWΦ)⁻¹ Φᵀ W r

And the minimum chi-squared (the **profile chi²**) is:

    χ²_profile = rᵀWr − bᵀc*   where b = ΦᵀWr

This is fast (a small matrix solve) and exact. No MCMC needed for the polynomial!

In [ ]:
# --- Let's do the calculation step by step for one order ---

# Step 1: Generate the normalised model
model_norm = sf._generate_stellar_model_ord(i_ord_example, **theta_test)

# Step 2: Extract valid (unmasked) pixels
mask = sf.ref_mask[i_ord_example]
data = sf.ref_spectrum[i_ord_example, ~mask]
sigma = sf.ref_uncert[i_ord_example, ~mask]
model = model_norm[~mask]

# Step 3: Work in log-flux space
log_residual = np.log(data) - np.log(model)

# Step 4: Propagate uncertainties (delta method: σ_log ≈ σ_flux / flux)
sigma_log = sigma / data
W = 1.0 / sigma_log**2

# Step 5: Get the Chebyshev basis matrix for this order.
# The basis is pre-computed on the FULL pixel grid (n_pixels × n_poly) so that
# the x-coordinate of each pixel is its position within the full order,
# consistently between fitting and plotting.
# We extract only the valid (unmasked) rows for the least-squares fit.
Phi_full = sf._cheb_bases[i_ord_example]   # shape: (n_pixels, n_poly)
Phi = Phi_full[~mask]                       # shape: (n_valid_pixels, n_poly)
print(f"Full basis shape  : {Phi_full.shape}  (one row per pixel in the order)")
print(f"Fitting basis shape: {Phi.shape}  (valid pixels only)")

In [ ]:
# Step 6: Build and solve the normal equations
WPhi = W[:, None] * Phi                    # weighted basis
A = Phi.T @ WPhi                            # (n_poly × n_poly) normal matrix
b = WPhi.T @ log_residual                   # (n_poly,) right-hand side
c_opt, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

print(f"Optimal polynomial coefficients: {c_opt}")
print(f"(T_0 ≈ {c_opt[0]:.4f} handles the overall normalisation)")

In [ ]:
# Step 7: Evaluate the polynomial on the full pixel grid and apply it.
# Since Phi_full is already on the full pixel grid, we just apply c_opt directly.
# This is consistent with how c_opt was derived (same x-coordinates).
log_poly = Phi_full @ c_opt
poly_correction = np.exp(log_poly)   # multiplicative correction in flux space

# Corrected model = raw model × polynomial (only at valid pixels)
model_corrected_out = np.full(n_pixels := sf.ref_wave.shape[-1], np.nan)
model_corrected_out[~mask] = model * np.exp(log_poly[~mask])

# --- Plot the result ---
wv = sf.ref_wave[i_ord_example]
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(wv, sf.ref_spectrum[i_ord_example], 'k', lw=0.7, label='Data')
axes[0].plot(wv, model_norm, 'r', lw=0.7, alpha=0.6, label='Model (before correction)')
axes[0].plot(wv, model_corrected_out, 'b', lw=0.7, alpha=0.9, label='Model × poly (after correction)')
axes[0].set_ylabel('Normalised flux')
axes[0].legend()
axes[0].set_title(f'Order #{i_ord_example} — effect of the polynomial correction')

axes[1].plot(wv, poly_correction, 'b', lw=1)
axes[1].axhline(1.0, color='k', linestyle='--', alpha=0.4)
axes[1].set_ylabel('Polynomial correction')
axes[1].set_xlabel('Wavelength (µm)')

plt.tight_layout()
plt.show()

In [ ]:
# Step 8: Compute the profile chi² and log-likelihood
chi2_profile = np.dot(W, log_residual**2) - np.dot(b, c_opt)

log_norm = -0.5 * np.sum(np.log(2.0 * np.pi * sigma_log**2))
logl = -0.5 * chi2_profile + log_norm

n_valid = np.sum(~mask)
print(f"Profile chi² = {chi2_profile:.1f}")
print(f"Reduced chi²  = {chi2_profile / (n_valid - sf.n_poly):.3f}  (should be close to 1 for a good fit)")
print(f"Profile log-likelihood = {logl:.1f}")
print()
print("The same calculation is repeated for all orders in profile_log_likelihood().")

---
## 6. Running the fit

### Step A: Quick point estimate with `run_minimize()`

We start with a fast minimisation to find the best-fit parameters.
This typically takes a few minutes on Narval with multiple CPUs.

**This step cannot be run locally** — it requires PHOENIX and PyAstronomy
which are only available on the cluster.

In [ ]:
# This call should be run on Narval (requires PHOENIX)
# result, theta_best, theta_dict_best = sf.run_minimize(n_restarts=5)

# For the tutorial, we load a pre-computed result instead:
# theta_dict_best = np.load('theta_best.npy', allow_pickle=True).item()

### Step B: Full posterior with `run_dynesty()`

After the point estimate confirms the fit is converging, we run dynesty
for the full posterior distribution. Dynesty is well suited for this
low-dimensional problem (~6 parameters) because:
- No walker initialisation or burn-in needed.
- Terminates automatically when the evidence estimate converges.
- Gives the Bayesian evidence Z, useful to compare models
  (e.g., to check whether adding a Gaussian prior on Teff improves the fit).

In [ ]:
# This call should also be run on Narval
# dynesty_results = sf.run_dynesty(n_live=400, save_file='dynesty_results.pkl')

---
## 7. Inspecting the result

Once the fit is done, we can use `make_model_with_best_poly()` to generate
the model spectra with the optimal polynomial applied to each order.
See the `analyse_stellar_fit.ipynb` notebook for a full analysis.

In [ ]:
# With a pre-loaded theta_dict, generate model + polynomial for all orders
# models_corrected = sf.make_model_with_best_poly(theta_dict_best)

# Example with our test parameters
models_corrected = sf.make_model_with_best_poly(sf.unpack_theta(sf.pack_theta(theta_test)))

n_orders = sf.ref_wave.shape[0]
fig, axes = plt.subplots(5, 1, figsize=(12, 12))

for ax, i_ord in zip(axes, range(0, n_orders, n_orders // 5)):
    wv = sf.ref_wave[i_ord]
    ax.plot(wv, sf.ref_spectrum[i_ord], 'k', lw=0.6, label='Data')
    ax.plot(wv, models_corrected[i_ord], 'r', lw=0.6, alpha=0.8, label='Model + poly')
    ax.set_ylabel(f'Order #{i_ord}')
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Wavelength (µm)')
plt.suptitle('Model vs. data after polynomial correction (test parameters)', y=1.01)
plt.tight_layout()
plt.show()

---
## Summary

| Step | What happens |
|---|---|
| `setup_stellar_fit()` | Load config, data, PHOENIX grid, pre-compute bases |
| `_generate_stellar_model_ord()` | PHOENIX interpolation → rotational broadening → RV shift → normalise |
| `_profile_logl_one_order()` | Solve for optimal polynomial analytically → compute profile χ² |
| `profile_log_likelihood()` | Sum over all orders |
| `lnprob()` | Add log prior |
| `run_minimize()` | Fast point estimate via scipy.optimize |
| `run_dynesty()` | Full posterior via nested sampling |
| `make_model_with_best_poly()` | Generate model with polynomial for visualisation |

The key idea is that the polynomial is **never a free parameter in the sampler** —
it is solved analytically at each evaluation, reducing the problem from
6 + N_orders × N_poly (potentially 6 + 31×4 = 130) parameters to just 6.
This makes the fit both fast and well-conditioned.